In [2]:
# imports, setup, and preprocessing

!pip install kagglehub -q

import kagglehub
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import random
from collections import Counter

# make folders for saving figures
os.makedirs('figures/task1', exist_ok=True)
os.makedirs('figures/task2', exist_ok=True)
os.makedirs('figures/task3', exist_ok=True)

# names for the 4 rul health categories
CATEGORY_NAMES = {
    0: 'Extremely Low RUL',
    1: 'Moderately Low RUL',
    2: 'Moderately High RUL',
    3: 'Extremely High RUL',
}

CAT_COLORS = {0: '#d62728', 1: '#ff7f0e', 2: '#2ca02c', 3: '#1f77b4'}


def assign_rul_category(rul_series):
    # split rul into 4 buckets using quantiles as the cutoffs
    q10 = rul_series.quantile(0.10)
    q50 = rul_series.quantile(0.50)
    q90 = rul_series.quantile(0.90)

    print(f'rul cutoffs: q10={q10:.2f}, q50={q50:.2f}, q90={q90:.2f}')

    labels = pd.Series(index=rul_series.index, dtype=int)
    labels[rul_series < q10]                             = 0
    labels[(rul_series >= q10) & (rul_series < q50)]     = 1
    labels[(rul_series >= q50) & (rul_series < q90)]     = 2
    labels[rul_series >= q90]                            = 3

    return labels


def load_and_preprocess(df_raw, nrows=10000):
    # just take the first 10000 rows as required
    df = df_raw.head(nrows).reset_index(drop=True)

    # grab all the sensor columns
    sensor_cols = sorted([c for c in df.columns if c.lower().startswith('sensor_')])

    # fill any missing values
    df[sensor_cols] = df[sensor_cols].apply(pd.to_numeric, errors='coerce')
    df[sensor_cols] = df[sensor_cols].ffill().bfill()

    # find the rul column
    rul_col = 'rul'
    if rul_col not in df.columns:
        matches = [c for c in df.columns if c.lower() == 'rul']
        if not matches:
            raise ValueError('no rul column found in the dataset')
        rul_col = matches[0]

    rul_labels = assign_rul_category(df[rul_col])
    return df, sensor_cols, rul_labels


# download the dataset and load the csv directly
print('downloading dataset from kaggle...')
path = kagglehub.dataset_download('anseldsouza/water-pump-rul-predictive-maintenance')

csv_path = os.path.join(path, 'rul_hrs.csv')
df_raw = pd.read_csv(csv_path)
print(f'raw dataset shape: {df_raw.shape}')

df, sensor_cols, rul_labels = load_and_preprocess(df_raw)
print(f'loaded {df.shape[0]} rows, {len(sensor_cols)} sensors')



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


downloading dataset from kaggle...


100%|██████████| 27.3M/27.3M [00:00<00:00, 38.2MB/s]

Extracting files...


raw dataset shape: (166441, 53)
rul cutoffs: q10=135.93, q50=202.59, q90=269.25
loaded 10000 rows, 50 sensors


In [2]:
# task 1: divide and conquer segmentation
# the idea: if a chunk of the signal has high variance, split it in half and check again
# if it's stable (low variance), keep it as one segment

RANDOM_SEED = 52

def divide_and_conquer(signal, lo, hi, threshold, segments):
    # base case: segment is too small or variance is low enough - keep it
    if hi <= lo:
        segments.append((lo, hi))
        return

    var = float(np.var(signal[lo:hi + 1]))

    if var <= threshold:
        # stable enough, mark it as a segment
        segments.append((lo, hi))
    else:
        # too noisy, split it down the middle and recurse
        mid = (lo + hi) // 2
        divide_and_conquer(signal, lo, mid, threshold, segments)
        divide_and_conquer(signal, mid + 1, hi, threshold, segments)


def segment_signal(signal, threshold):
    segments = []
    divide_and_conquer(signal, 0, len(signal) - 1, threshold, segments)
    return segments


def plot_segmentation(signal, rul_labels, segments, sensor_name, score, save_path):
    fig, ax = plt.subplots(figsize=(14, 4))

    # shade each segment by the dominant rul category in that range
    for (lo, hi) in segments:
        seg_labels = rul_labels.iloc[lo:hi + 1]
        dominant = int(seg_labels.mode()[0]) if len(seg_labels) > 0 else 0
        ax.axvspan(lo, hi, alpha=0.18, color=CAT_COLORS[dominant], linewidth=0)

    # draw a dashed line at each segment boundary
    seen = set()
    for (lo, _) in segments:
        if lo not in seen:
            ax.axvline(lo, color='grey', linewidth=0.6, linestyle='--', alpha=0.7)
            seen.add(lo)

    ax.plot(signal, color='black', linewidth=0.7, alpha=0.85)

    patches = [mpatches.Patch(color=CAT_COLORS[k], alpha=0.5, label=CATEGORY_NAMES[k])
               for k in sorted(CAT_COLORS)]
    ax.legend(handles=patches, loc='upper right', fontsize=7)
    ax.set_title(f'{sensor_name}  |  complexity score = {score}', fontsize=11)
    ax.set_xlabel('time index')
    ax.set_ylabel('sensor value')
    plt.tight_layout()
    plt.savefig(save_path, dpi=120)
    plt.show()
    plt.close()


def plot_complexity_bar(complexity_scores):
    sensors = list(complexity_scores.keys())
    scores = [complexity_scores[s] for s in sensors]

    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.bar(sensors, scores, color='#4c72b0', edgecolor='white')
    ax.bar_label(bars, fontsize=8)
    ax.set_title('segmentation complexity scores - task 1')
    ax.set_xlabel('sensor')
    ax.set_ylabel('number of segments')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    path = 'figures/task1/complexity_scores_bar.png'
    plt.savefig(path, dpi=120)
    plt.show()
    plt.close()


def run_task1(df, sensor_cols, rul_labels):
    # toy example first just to verify the algorithm works
    print('toy example for task 1:')
    toy = np.array([1.0, 1.1, 1.0, 1.1, 10.0, 0.0, 10.0, 0.0])
    toy_segs = segment_signal(toy, threshold=1.0)
    print(f'  signal: {toy}')
    print(f'  segments found: {toy_segs}')
    print(f'  complexity score: {len(toy_segs)}')

    # pick 10 sensors randomly
    rng = random.Random(RANDOM_SEED)
    selected = sorted(rng.sample(sensor_cols, min(10, len(sensor_cols))))
    print(f'\nselected sensors: {selected}')

    complexity_scores = {}

    for sensor in selected:
        signal = df[sensor].values.astype(float)

        # set threshold as the 25th percentile of rolling variances
        rolling_vars = pd.Series(signal).rolling(window=50, min_periods=1).var().dropna().values
        threshold = float(np.percentile(rolling_vars, 25)) if len(rolling_vars) > 0 else float(np.var(signal) * 0.05)

        # make sure threshold isn't zero
        if threshold == 0:
            threshold = max(float(np.var(signal) * 0.01), 1e-9)

        segments = segment_signal(signal, threshold)
        score = len(segments)
        complexity_scores[sensor] = score

        save_path = f'figures/task1/{sensor}_segmentation.png'
        plot_segmentation(signal, rul_labels, segments, sensor, score, save_path)
        print(f'  {sensor}: {score} segments')

    plot_complexity_bar(complexity_scores)

    print('\ncomplexity scores summary:')
    for s, sc in sorted(complexity_scores.items(), key=lambda x: -x[1]):
        print(f'  {s}: {sc}')

    return complexity_scores


print('\n--- task 1: segmentation ---')
run_task1(df, sensor_cols, rul_labels)


--- task 1: segmentation ---
toy example for task 1:
  signal: [ 1.   1.1  1.   1.1 10.   0.  10.   0. ]
  segments found: [(0, 3), (4, 4), (5, 5), (6, 6), (7, 7)]
  complexity score: 5

selected sensors: ['sensor_02', 'sensor_03', 'sensor_08', 'sensor_10', 'sensor_18', 'sensor_24', 'sensor_27', 'sensor_31', 'sensor_33', 'sensor_47']
  sensor_02: 1155 segments
  sensor_03: 615 segments
  sensor_08: 2030 segments
  sensor_10: 883 segments
  sensor_18: 3149 segments
  sensor_24: 2676 segments
  sensor_27: 2358 segments
  sensor_31: 1862 segments
  sensor_33: 1464 segments
  sensor_47: 1165 segments

complexity scores summary:
  sensor_18: 3149
  sensor_24: 2676
  sensor_27: 2358
  sensor_08: 2030
  sensor_31: 1862
  sensor_33: 1464
  sensor_47: 1165
  sensor_02: 1155
  sensor_10: 883
  sensor_03: 615


{'sensor_02': 1155,
 'sensor_03': 615,
 'sensor_08': 2030,
 'sensor_10': 883,
 'sensor_18': 3149,
 'sensor_24': 2676,
 'sensor_27': 2358,
 'sensor_31': 1862,
 'sensor_33': 1464,
 'sensor_47': 1165}

In [4]:
# cell 3 - task 2: divide and conquer clustering
# start with everything in one cluster, then keep splitting
# the most spread-out cluster until we have 4 total

K = 4


def intra_cluster_variance(points):
    # basically just how spread out the points are across all dimensions
    if len(points) == 0:
        return 0.0
    return float(np.mean(np.var(points, axis=0)))


def split_cluster(points, indices):
    # find the dimension with the most spread and split there
    split_dim = int(np.argmax(np.var(points, axis=0)))
    median_val = float(np.median(points[:, split_dim]))

    left_mask = points[:, split_dim] <= median_val
    right_mask = ~left_mask

    # edge case: if everything lands on one side, just split by index
    if left_mask.sum() == 0 or right_mask.sum() == 0:
        half = len(points) // 2
        left_mask = np.zeros(len(points), dtype=bool)
        left_mask[:half] = True
        right_mask = ~left_mask

    return (points[left_mask], indices[left_mask],
            points[right_mask], indices[right_mask])


def divide_conquer_cluster(data, k):
    N = data.shape[0]
    labels = np.zeros(N, dtype=int)

    # start with one big cluster containing everything
    clusters = [(data, np.arange(N))]

    while len(clusters) < k:
        # pick the cluster with the highest variance to split next
        variances = [intra_cluster_variance(c[0]) for c in clusters]
        worst = int(np.argmax(variances))

        pts, idxs = clusters.pop(worst)

        if len(pts) < 2:
            # can't split a single point, put it back and stop
            clusters.append((pts, idxs))
            break

        lp, li, rp, ri = split_cluster(pts, idxs)
        clusters.append((lp, li))
        clusters.append((rp, ri))

    # assign integer labels to each cluster
    for cid, (_, idxs) in enumerate(clusters):
        labels[idxs] = cid

    return labels


def zscore_normalise(arr):
    # z-score each column so no single sensor dominates just because of scale
    means = arr.mean(axis=0)
    stds = arr.std(axis=0)
    stds[stds == 0] = 1.0
    return (arr - means) / stds


def majority_class_analysis(cluster_labels, rul_labels):
    rul_arr = rul_labels.values
    result = {}

    print(f"  {'cluster':<10} {'dominant class':<26} {'count':>7} {'total':>7} {'purity':>8}")
    print(f"  {'-' * 60}")

    for cid in range(K):
        mask = cluster_labels == cid
        subset = rul_arr[mask]
        if len(subset) == 0:
            result[cid] = {}
            continue

        counts = Counter(subset.tolist())
        dom = max(counts, key=counts.get)
        purity = 100.0 * counts[dom] / len(subset)
        result[cid] = dict(counts)
        print(f"  cluster {cid:<3} {CATEGORY_NAMES[dom]:<26} {counts[dom]:>7} {len(subset):>7} {purity:>7.1f}%")

    return result


def plot_cluster_heatmap(analysis, save_path):
    # stacked bar showing how much of each cluster is each rul category
    matrix = np.zeros((K, 4))
    for cid, counts in analysis.items():
        total = sum(counts.values()) or 1
        for cls, cnt in counts.items():
            matrix[cid, int(cls)] = cnt / total  # int(cls) fixes the float index issue

    fig, ax = plt.subplots(figsize=(8, 5))
    bottom = np.zeros(K)
    colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4']

    for cls in range(4):
        ax.bar(range(K), matrix[:, cls], bottom=bottom,
               color=colors[cls], label=CATEGORY_NAMES[cls], edgecolor='white')
        bottom += matrix[:, cls]

    ax.set_xticks(range(K))
    ax.set_xticklabels([f'cluster {i}' for i in range(K)])
    ax.set_ylabel('fraction of points')
    ax.set_title('rul category distribution per cluster')
    ax.legend(loc='upper right', fontsize=8)
    plt.tight_layout()
    plt.savefig(save_path, dpi=120)
    plt.show()
    plt.close()


def plot_pca_scatter(data, cluster_labels, rul_labels, save_path):
    # manual pca - just project onto the top 2 principal components
    centered = data - data.mean(axis=0)
    cov = centered.T @ centered / (len(centered) - 1)
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = np.argsort(eigvals)[::-1]
    pc = centered @ eigvecs[:, order[:2]]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    clust_colors = ['#e377c2', '#17becf', '#bcbd22', '#9467bd']
    rul_arr = rul_labels.values

    for cid in range(K):
        mask = cluster_labels == cid
        axes[0].scatter(pc[mask, 0], pc[mask, 1], color=clust_colors[cid],
                        alpha=0.3, s=5, label=f'cluster {cid}')
    axes[0].set_title('pca - by cluster')
    axes[0].legend(fontsize=8)
    axes[0].set_xlabel('pc1')
    axes[0].set_ylabel('pc2')

    for cls in range(4):
        mask = rul_arr == cls
        axes[1].scatter(pc[mask, 0], pc[mask, 1], color=CAT_COLORS[cls],
                        alpha=0.3, s=5, label=CATEGORY_NAMES[cls])
    axes[1].set_title('pca - by true rul category')
    axes[1].legend(fontsize=7)
    axes[1].set_xlabel('pc1')
    axes[1].set_ylabel('pc2')

    plt.suptitle('task 2 - clustering vs true rul categories', fontsize=12)
    plt.tight_layout()
    plt.savefig(save_path, dpi=120)
    plt.show()
    plt.close()


def run_task2(df, sensor_cols, rul_labels):
    # toy example first
    print('toy example for task 2:')
    toy = np.array([[1.0, 1.0], [1.1, 0.9], [0.9, 1.1],
                    [9.0, 9.0], [9.1, 8.9], [8.9, 9.1]])
    toy_labels = divide_conquer_cluster(toy, k=2)
    print(f'  points:\n{toy}')
    print(f'  labels: {toy_labels}  (should be two groups)')

    print('\npreparing feature matrix...')
    data = zscore_normalise(df[sensor_cols].values.astype(float))

    print(f'running clustering with k={K}...')
    cluster_labels = divide_conquer_cluster(data, k=K)

    print('\ncluster sizes:')
    for cid in range(K):
        print(f'  cluster {cid}: {(cluster_labels == cid).sum()} points')

    analysis = majority_class_analysis(cluster_labels, rul_labels)

    heatmap_path = 'figures/task2/cluster_rul_heatmap.png'
    plot_cluster_heatmap(analysis, heatmap_path)

    pca_path = 'figures/task2/cluster_pca_scatter.png'
    plot_pca_scatter(data, cluster_labels, rul_labels, pca_path)

    return cluster_labels, analysis


print('\n--- task 2: clustering ---')
run_task2(df, sensor_cols, rul_labels)


--- task 2: clustering ---
toy example for task 2:
  points:
[[1.  1. ]
 [1.1 0.9]
 [0.9 1.1]
 [9.  9. ]
 [9.1 8.9]
 [8.9 9.1]]
  labels: [0 0 0 1 1 1]  (should be two groups)

preparing feature matrix...
running clustering with k=4...

cluster sizes:
  cluster 0: 5685 points
  cluster 1: 2157 points
  cluster 2: 1079 points
  cluster 3: 1079 points
  cluster    dominant class               count   total   purity
  ------------------------------------------------------------
  cluster 0   Moderately High RUL           2343    5685    41.2%
  cluster 1   Moderately Low RUL            1422    2157    65.9%
  cluster 2   Moderately High RUL            648    1079    60.1%
  cluster 3   Moderately High RUL            542    1079    50.2%


(array([0, 0, 0, ..., 0, 0, 0]),
 {0: {3.0: 595, 2.0: 2343, 1.0: 2142, 0.0: 605},
  1: {3.0: 47, 2.0: 467, 1.0: 1422, 0.0: 221},
  2: {3.0: 240, 2.0: 648, 1.0: 117, 0.0: 74},
  3: {3.0: 118, 2.0: 542, 1.0: 319, 0.0: 100}})

In [5]:
# cell 4 - task 3: kadane's algorithm
# for each sensor, find the interval with the most sustained deviation
# then check what rul category that interval mostly falls in


def kadane(x):
    # standard kadane - finds the subarray with the max sum in O(n)
    if len(x) == 0:
        return 0.0, 0, 0

    max_sum = float(x[0])
    current_sum = float(x[0])
    start = 0
    temp_start = 0
    end = 0

    for i in range(1, len(x)):
        if current_sum + x[i] > x[i]:
            current_sum += x[i]
        else:
            # starting fresh is better here
            current_sum = x[i]
            temp_start = i

        if current_sum > max_sum:
            max_sum = current_sum
            start = temp_start
            end = i

    return max_sum, start, end


def compute_deviation_signal(signal):
    # absolute first difference, then centre it around zero
    d = np.abs(np.diff(signal.astype(float)))
    return d - d.mean()


def interval_rul_counts(rul_labels, start, end):
    # map back to original indices (diff shifts everything by 1)
    orig_end = min(end + 1, len(rul_labels) - 1)
    subset = rul_labels.iloc[start:orig_end + 1]
    return subset.value_counts().to_dict()


def analyse_sensor(signal, rul_labels, sensor_name):
    x = compute_deviation_signal(signal)
    max_sum, s, e = kadane(x)
    counts = interval_rul_counts(rul_labels, s, e)
    dom_cat = max(counts, key=counts.get) if counts else -1
    rel_pos = s / max(len(x) - 1, 1)

    return {
        'sensor': sensor_name,
        'max_sum': max_sum,
        'start': s,
        'end': e,
        'length': e - s + 1,
        'rel_pos': rel_pos,
        'rul_counts': counts,
        'dominant_cat': dom_cat,
    }


def plot_sensor_kadane(x, result, rul_labels, save_path):
    fig, ax = plt.subplots(figsize=(14, 4))

    # shade background by rul category
    rul_arr = rul_labels.values
    step = max(1, len(x) // 500)
    for i in range(0, len(x), step):
        hi_i = min(i + step, len(x) - 1)
        ax.axvspan(i, hi_i, alpha=0.12, color=CAT_COLORS[int(rul_arr[i])], linewidth=0)

    ax.plot(x, color='steelblue', linewidth=0.7, alpha=0.9, label='deviation signal')

    # highlight the max subarray in red
    s, e = result['start'], result['end']
    ax.axvspan(s, e, alpha=0.45, color='red', label=f'max interval [{s},{e}]')
    ax.plot(range(s, e + 1), x[s:e + 1], color='darkred', linewidth=1.2)

    patches = [mpatches.Patch(color=CAT_COLORS[k], alpha=0.5, label=CATEGORY_NAMES[k])
               for k in sorted(CAT_COLORS)]
    patches.append(mpatches.Patch(color='red', alpha=0.5, label='max deviation interval'))
    ax.legend(handles=patches, loc='upper right', fontsize=7)

    dom = result['dominant_cat']
    ax.set_title(
        f"{result['sensor']}  |  max_sum={result['max_sum']:.3f}  "
        f"interval=[{s},{e}]  dominant rul: {CATEGORY_NAMES.get(dom, '?')}",
        fontsize=9
    )
    ax.set_xlabel('time index (after diff)')
    ax.set_ylabel('centred abs-diff')
    plt.tight_layout()
    plt.savefig(save_path, dpi=120)
    plt.show()
    plt.close()


def plot_summary_heatmap(results, save_path):
    # heatmap: what fraction of each sensor's max interval falls in each rul category
    sensor_names = [r['sensor'] for r in results]
    matrix = np.zeros((len(results), 4))

    for i, r in enumerate(results):
        total = sum(r['rul_counts'].values()) or 1
        for cls, cnt in r['rul_counts'].items():
            matrix[i, int(cls)] = cnt / total

    fig, ax = plt.subplots(figsize=(8, max(6, len(results) * 0.35)))
    im = ax.imshow(matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
    ax.set_xticks(range(4))
    ax.set_xticklabels([CATEGORY_NAMES[k] for k in range(4)], rotation=20, ha='right', fontsize=8)
    ax.set_yticks(range(len(sensor_names)))
    ax.set_yticklabels(sensor_names, fontsize=7)
    ax.set_title('fraction of max-deviation interval in each rul category')
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.04)
    plt.tight_layout()
    plt.savefig(save_path, dpi=130)
    plt.show()
    plt.close()


def plot_early_warning_bar(results, save_path):
    # rank sensors by how much of their deviation interval is in low-rul and how early it appears
    scored = []
    for r in results:
        total = sum(r['rul_counts'].values()) or 1
        low_frac = (r['rul_counts'].get(0, 0) + r['rul_counts'].get(1, 0)) / total
        ew_score = low_frac / (1.0 + r['rel_pos'])
        scored.append((r['sensor'], ew_score, low_frac))

    scored.sort(key=lambda t: -t[1])
    sensors = [t[0] for t in scored]
    ew_scores = [t[1] for t in scored]
    low_fracs = [t[2] for t in scored]

    x = np.arange(len(sensors))
    width = 0.4
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(x - width / 2, ew_scores, width, label='early warning score', color='#d62728')
    ax.bar(x + width / 2, low_fracs, width, label='fraction in low-rul', color='#ff7f0e')
    ax.set_xticks(x)
    ax.set_xticklabels(sensors, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('score')
    ax.set_title('task 3 - sensor early warning ranking')
    ax.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=120)
    plt.show()
    plt.close()


def run_task3(df, sensor_cols, rul_labels):
    # toy example first
    print('toy example for task 3 (kadane):')
    toy = np.array([-2.0, 1, -3, 4, -1, 2, 1, -5, 4])
    ms, s, e = kadane(toy)
    print(f'  array: {toy}')
    print(f'  result: sum={ms}, start={s}, end={e}, subarray={toy[s:e+1]}')
    print(f'  expected: sum=6, indices 3-6, subarray=[4 -1 2 1]')

    results = []
    print(f"\n  {'sensor':<14} {'max_sum':>10} {'start':>7} {'end':>7} {'len':>6}  dominant rul")
    print(f"  {'-' * 65}")

    for sensor in sensor_cols:
        signal = df[sensor].values.astype(float)
        r = analyse_sensor(signal, rul_labels, sensor)
        results.append(r)

        dom_name = CATEGORY_NAMES.get(r['dominant_cat'], '?')
        print(f"  {sensor:<14} {r['max_sum']:>10.3f} {r['start']:>7} {r['end']:>7} {r['length']:>6}  {dom_name}")

        x = compute_deviation_signal(signal)
        save_path = f"figures/task3/{sensor}_kadane.png"
        plot_sensor_kadane(x, r, rul_labels, save_path)

    plot_summary_heatmap(results, 'figures/task3/task3_rul_heatmap.png')
    plot_early_warning_bar(results, 'figures/task3/task3_early_warning_rank.png')

    # print the top 5 sensors that are best at flagging low rul early
    print('\ntop 5 early warning sensors:')
    scored = sorted(
        results,
        key=lambda r: (r['rul_counts'].get(0, 0) + r['rul_counts'].get(1, 0))
                      / max(sum(r['rul_counts'].values()), 1) / (1 + r['rel_pos']),
        reverse=True
    )
    for rank, r in enumerate(scored[:5], 1):
        total = sum(r['rul_counts'].values()) or 1
        low_frac = (r['rul_counts'].get(0, 0) + r['rul_counts'].get(1, 0)) / total
        print(f"  {rank}. {r['sensor']} - {100 * low_frac:.1f}% of interval in low rul (position={r['rel_pos']:.3f})")

    return results


print('\n--- task 3: kadane ---')
run_task3(df, sensor_cols, rul_labels)


--- task 3: kadane ---
toy example for task 3 (kadane):
  array: [-2.  1. -3.  4. -1.  2.  1. -5.  4.]
  result: sum=6.0, start=3, end=6, subarray=[ 4. -1.  2.  1.]
  expected: sum=6, indices 3-6, subarray=[4 -1 2 1]

  sensor            max_sum   start     end    len  dominant rul
  -----------------------------------------------------------------
  sensor_00           9.494      62    4830   4769  Moderately High RUL
  sensor_01         184.005       1    2038   2038  Moderately High RUL
  sensor_02          19.574    8990    9825    836  Extremely Low RUL
  sensor_03          32.339    5217    9488   4272  Moderately Low RUL
  sensor_04        1042.733     736    5914   5179  Moderately High RUL
  sensor_05         859.843     102    4789   4688  Moderately High RUL
  sensor_06           7.656    2689    4488   1800  Moderately High RUL
  sensor_07          11.365    3884    9996   6113  Moderately Low RUL
  sensor_08          24.515    5767    8749   2983  Moderately Low RUL
  sen

[{'sensor': 'sensor_00',
  'max_sum': np.float64(9.493931137414075),
  'start': 62,
  'end': 4830,
  'length': 4769,
  'rel_pos': 0.00620124024804961,
  'rul_counts': {2.0: 3832, 3.0: 938},
  'dominant_cat': 2.0},
 {'sensor': 'sensor_01',
  'max_sum': np.float64(184.00498048484283),
  'start': 1,
  'end': 2038,
  'length': 2038,
  'rel_pos': 0.00010002000400080016,
  'rul_counts': {2.0: 1040, 3.0: 999},
  'dominant_cat': 2.0},
 {'sensor': 'sensor_02',
  'max_sum': np.float64(19.57380504361334),
  'start': 8990,
  'end': 9825,
  'length': 836,
  'rel_pos': 0.8991798359671934,
  'rul_counts': {0.0: 827, 1.0: 10},
  'dominant_cat': 0.0},
 {'sensor': 'sensor_03',
  'max_sum': np.float64(32.33858125479592),
  'start': 5217,
  'end': 9488,
  'length': 4272,
  'rel_pos': 0.5218043608721744,
  'rul_counts': {1.0: 3783, 0.0: 490},
  'dominant_cat': 1.0},
 {'sensor': 'sensor_04',
  'max_sum': np.float64(1042.733489448872),
  'start': 736,
  'end': 5914,
  'length': 5179,
  'rel_pos': 0.073614722